# CerberusAI v2 - Entrenamiento QLoRA en Google Colab (corrida controlada)

Re-entrena **desde cero** `Qwen/Qwen2.5-Coder-7B-Instruct` con QLoRA 4-bit sobre
`v2 + v3 + v4_balance` (1594 ejemplos). Cada corrida vive en su propia carpeta de Drive
`MyDrive/CerberusAI/runs/<RUN_ID>/` con:

- `run_info.json`: commit, GPU, versiones, hiperparametros, sha256 de los datos y conteos.
- `train.jsonl` / `eval.jsonl`: copia exacta de los datos usados.
- `train.log`: salida completa del entrenamiento.
- `lora-adapter/`: adapter final (mejor `eval_loss`), checkpoints y `log_history.json`.

**Ejecuta las celdas en orden, una por una.** Cada celda verifica su resultado con `assert` y
se detiene si algo no cuadra. Runtime: **GPU L4 o A100** (Entorno de ejecucion > Cambiar tipo).

## 0. Parametros de la corrida

- Corrida nueva: pon un `RUN_ID` que no exista y deja `RESUME = False`.
- Si la sesion se cayo a mitad del entrenamiento: **mismo** `RUN_ID`, `RESUME = True`, y vuelve a
  ejecutar todas las celdas en orden (continua desde el ultimo checkpoint en Drive).

In [ ]:
RUN_ID = "v2-20261106-1"   # identificador unico de esta corrida
RESUME = False             # True solo para retomar ESTE RUN_ID tras una desconexion

REPO = "https://github.com/Coliiin1/CerberusIDE-IA.git"
BRANCH = "main"
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
EXPECTED_TOTAL = 1594

# Hiperparametros identicos a la corrida 1 (v1).
HP = {"epochs": 5, "batch": 2, "grad-accum": 4, "lr": 2e-4,
      "lora-r": 16, "lora-alpha": 32, "max-seq-length": 2048, "seed": 42}

DATASETS = [
    "dataset/cerberus_dataset_v2.jsonl",
    "dataset/cerberus_dataset_v3_valido.jsonl",
    "dataset/cerberus_dataset_v3_errores_semanticos.jsonl",
    "dataset/cerberus_dataset_v3_errores_sintacticos.jsonl",
    "dataset/cerberus_dataset_v4_balance.jsonl",
]

REPO_DIR = "/content/CerberusIDE-IA"
AI_DIR = REPO_DIR + "/CerberusAI"
RUN_DIR = "/content/drive/MyDrive/CerberusAI/runs/" + RUN_ID
ADAPTER = RUN_DIR + "/lora-adapter"
print("RUN_ID:", RUN_ID, "| RESUME:", RESUME)
print("RUN_DIR:", RUN_DIR)

## 1. Verificar que hay GPU

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "Sin GPU: Entorno de ejecucion > Cambiar tipo de entorno > L4 o A100"
assert "T4" not in r.stdout, "Te toco una T4 (16 GB, sin bf16). Cambia a L4 o A100."

## 2. Montar Google Drive y comprobar la carpeta de la corrida

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

if RESUME:
    assert os.path.isfile(RUN_DIR + "/run_info.json"), "RESUME=True pero la corrida no existe: " + RUN_DIR
    assert not os.path.isfile(ADAPTER + "/adapter_model.safetensors"), "Esta corrida ya termino; no hay nada que reanudar."
    print("Reanudando corrida existente. Checkpoints:",
          sorted(d for d in os.listdir(ADAPTER) if d.startswith("checkpoint-")))
else:
    assert not os.path.exists(RUN_DIR), RUN_DIR + " ya existe: usa otro RUN_ID (o RESUME=True para continuar)."
    print("OK: corrida nueva, la carpeta se creara en el paso 5.")

## 3. Clonar el repositorio

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO, REPO_DIR], check=True)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
os.chdir(AI_DIR)
print("branch:", BRANCH, "| commit:", COMMIT)
print("cwd:", os.getcwd())

## 4. Instalar dependencias y verificar versiones

Usa el `torch` CUDA de Colab (no se reinstala). Si el `assert` de torch falla, ve a
**Entorno de ejecucion > Reiniciar sesion** y vuelve a ejecutar desde la celda 0.

In [ ]:
!pip install -q -r /content/CerberusIDE-IA/CerberusAI/colab/requirements-colab.txt

In [ ]:
from importlib.metadata import version
import torch
VERSIONS = {p: version(p) for p in
            ["torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets"]}
for k, v in VERSIONS.items():
    print("{:<14} {}".format(k, v))
assert VERSIONS["torch"] == torch.__version__, "pip cambio torch: reinicia la sesion y vuelve a la celda 0."
assert torch.cuda.is_available()
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| bf16:", torch.cuda.is_bf16_supported())
assert torch.cuda.is_bf16_supported(), "La GPU no soporta bf16; usa L4 o A100."

## 5. Construir y registrar los datos

`to_chat` + `split` (estratificado, seed 42) + `--check`. Escribe `run_info.json` y copia
`train.jsonl` / `eval.jsonl` a la carpeta de la corrida. Al reanudar, comprueba que los datos
son identicos a los de la corrida original.

In [ ]:
import collections, datetime, hashlib, json, os, shutil, subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout, r.stderr)
    assert r.returncode == 0, "fallo: " + " ".join(cmd)

def sha256(p):
    with open(p, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

def contar(p):
    with open(p, encoding="utf-8") as f:
        return collections.Counter(json.loads(l)["category"] for l in f if l.strip())

run([sys.executable, "finetune/to_chat.py", "--input", *DATASETS, "--output", "finetune/chat_all.jsonl"])
run([sys.executable, "finetune/split.py", "--input", "finetune/chat_all.jsonl", "--out-dir", "finetune",
     "--eval-ratio", "0.2", "--seed", str(HP["seed"])])
run([sys.executable, "finetune/train_qlora.py", "--check"])

cats = contar("finetune/chat_all.jsonl")
total = sum(cats.values())
assert total == EXPECTED_TOTAL, "se esperaban {} ejemplos y hay {}".format(EXPECTED_TOTAL, total)
n_train = sum(contar("finetune/train.jsonl").values())
n_eval = sum(contar("finetune/eval.jsonl").values())
assert n_train + n_eval == total

datos = {"datasets": {d: sha256(d) for d in DATASETS},
         "train.jsonl": sha256("finetune/train.jsonl"),
         "eval.jsonl": sha256("finetune/eval.jsonl")}

if RESUME:
    with open(RUN_DIR + "/run_info.json", encoding="utf-8") as f:
        previo = json.load(f)
    assert previo["sha256"] == datos, "Los datos cambiaron respecto a la corrida original: no se puede reanudar."
    print("Datos identicos a la corrida original.")
else:
    os.makedirs(ADAPTER)
    shutil.copy("finetune/train.jsonl", RUN_DIR + "/train.jsonl")
    shutil.copy("finetune/eval.jsonl", RUN_DIR + "/eval.jsonl")
    run_info = {
        "run_id": RUN_ID,
        "fecha_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "repo": REPO, "branch": BRANCH, "commit": COMMIT,
        "modelo_base": MODEL, "gpu": GPU, "versiones": VERSIONS, "hiperparametros": HP,
        "ejemplos": {"total": total, "train": n_train, "eval": n_eval},
        "categorias": dict(sorted(cats.items())),
        "sha256": datos,
    }
    with open(RUN_DIR + "/run_info.json", "w", encoding="utf-8") as f:
        json.dump(run_info, f, indent=2, ensure_ascii=False)
    print(json.dumps(run_info, indent=2, ensure_ascii=False))

## 6. Entrenar

Unos 800 pasos (5 epocas). Los checkpoints se guardan por epoca en Drive y la salida va a
`train.log`. Al final se conserva el adapter con menor `eval_loss`.

In [ ]:
import shlex, sys, os
cmd = [sys.executable, "-u", "finetune/train_qlora.py", "--model", MODEL, "--output", ADAPTER, "--dtype", "auto"]
for k, v in HP.items():
    cmd += ["--" + k, str(v)]
if RESUME:
    cmd.append("--resume")
CMD = shlex.join(cmd)
LOG = shlex.quote(RUN_DIR + "/train.log")
print(CMD)
!{CMD} 2>&1 | tee -a {LOG}
assert os.path.isfile(ADAPTER + "/adapter_model.safetensors"), "No se genero el adapter: revisa train.log"
assert os.path.isfile(ADAPTER + "/log_history.json")
print("Entrenamiento terminado.")

## 7. Resumen de metricas

In [ ]:
import json
with open(ADAPTER + "/log_history.json", encoding="utf-8") as f:
    h = json.load(f)
print("mejor checkpoint:", h["best_model_checkpoint"], "| mejor eval_loss:", h["best_metric"])
print("{:>6} {:>12} {:>12}".format("epoca", "train_loss", "eval_loss"))
ultimo_train = None
for e in h["log_history"]:
    if "loss" in e:
        ultimo_train = e["loss"]
    if "eval_loss" in e:
        print("{:>6.2f} {:>12} {:>12.4f}".format(e["epoch"], ultimo_train, e["eval_loss"]))

## 8. Empaquetar y descargar

Zip con el adapter final (sin `checkpoint-*`) + `run_info.json`. Tambien queda en Drive.

In [ ]:
import os, zipfile
ZIP = "{}/lora-adapter-{}.zip".format(RUN_DIR, RUN_ID)
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir(ADAPTER)):
        p = os.path.join(ADAPTER, name)
        if os.path.isfile(p):  # los checkpoint-* son carpetas y se omiten
            z.write(p, "lora-adapter/" + name)
    z.write(RUN_DIR + "/run_info.json", "run_info.json")
    z.write(RUN_DIR + "/train.log", "train.log")
print(ZIP, round(os.path.getsize(ZIP) / 1e6, 1), "MB")
with zipfile.ZipFile(ZIP) as z:
    print("\n".join(z.namelist()))
from google.colab import files
files.download(ZIP)

## 9. Siguiente: merge / GGUF / Ollama / evaluacion (en local)

1. Backup de v1: `CerberusAI/finetune/lora-adapter` -> `lora-adapter-v1` y
   `cerberusai-q8_0.gguf` -> `cerberusai-v1-q8_0.gguf` (el modelo `cerberusai` de Ollama no se toca).
2. Descomprime el zip en `CerberusAI/finetune/` (crea `lora-adapter/`).
3. `py CerberusAI\finetune\export_gguf.py` -> `convert_hf_to_gguf.py` -> `llama-quantize Q8_0`.
4. `ollama create cerberusai-v2 -f CerberusAI\finetune\Modelfile` (con `FROM` al GGUF nuevo).
5. Regenera `chat_all/train/eval.jsonl` en local y compara sus sha256 con `run_info.json`.
6. `py CerberusAI\eval\run_eval.py --base-model cerberusai --ft-model cerberusai-v2` y
   `py CerberusAI\eval\run_eval.py --ft-model cerberusai-v2`.